## 电信用户流失预测

### 第一步： 加载与清洗数据

In [2]:
import pandas as pd
import numpy as np

# 了解数据集的基本情况

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

'''
对数据集进行观察
print(df.shape)
print(df.info)
print(df.head(10))
print(df.isna().sum())
'''
print(df.shape)
print(df.dtypes)
print(df.head(3))

(7043, 21)
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProt

In [3]:
#发现 TotalCharges字段不是数字类型，但又没有空值，进行强制转换，不成功就直接转换为NAN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"],errors='coerce')
'''
对转换后的数据进行调试和观察
# print(df.isna().sum())
# missing_rows = df.loc[df['TotalCharges'].isna()]
# print(f"实际发现的空行数量: {len(missing_rows)}")
'''

#发现 TotalCharges 字段仍然有NAN值，将其调整成为 0， 方便进行后续计算
df["TotalCharges"] = df["TotalCharges"].fillna(0)
'''
继续对调整后的数据进行调试和观察
missing_rows = df.loc[df['TotalCharges'].isna()]
print(f"实际发现的空行数量: {len(missing_rows)}")
print(missing_rows)
print(f"当前数据类型: {df['TotalCharges'].dtype}")
'''


'\n继续对调整后的数据进行调试和观察\nmissing_rows = df.loc[df[\'TotalCharges\'].isna()]\nprint(f"实际发现的空行数量: {len(missing_rows)}")\nprint(missing_rows)\nprint(f"当前数据类型: {df[\'TotalCharges\'].dtype}")\n'

In [4]:
# 数据集中有的customerID 字段，是无效的特征，予以去除
df = df.drop('customerID', axis=1)

# 数据集中的Churn字段是目标变量，但现在是分类变量，需要转换成数字变量，才能喂给scikit-learn进行学习
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print(f"数据清理工作完成, 总样本数{df.shape[0]}")

数据清理工作完成, 总样本数7043


In [17]:
print(df.head(3))

   gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  Female              0     Yes         No       1           No   
1    Male              0      No         No      34          Yes   
2    Male              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity OnlineBackup  \
0  No phone service             DSL             No          Yes   
1                No             DSL            Yes           No   
2                No             DSL            Yes          Yes   

  DeviceProtection TechSupport StreamingTV StreamingMovies        Contract  \
0               No          No          No              No  Month-to-month   
1              Yes          No          No              No        One year   
2               No          No          No              No  Month-to-month   

  PaperlessBilling     PaymentMethod  MonthlyCharges  TotalCharges  Churn  
0              Yes  Electronic check           29.85         29.85   

### 第二步：防泄露切分与特征分组

In [5]:
from sklearn.model_selection import train_test_split
# 将特征值和预测值划分出来
X = df.drop("Churn", axis = 1)
y = df["Churn"]

# 按照20%的比例划分训练集与测试集，并保证两类数据集的流失比例一致，即stratify = y 
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42, test_size=0.2, stratify=y)

# 将数字类型特征和类别特征分别提取出来
numeric_features = X_train.select_dtypes(include = ["int64", "float64"]).columns.tolist()
category_features = X_train.select_dtypes(include = ["object"]).columns.tolist()
print(f"数值类型特征供{len(numeric_features)}列, 分类类型特征共{len(category_features)}列")

数值类型特征供4列, 分类类型特征共15列


### 第三步：进行数据处理

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# 搭建数据处理管线，对数值列进行转换
num_transformer = Pipeline(steps = [
    ("imupter", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())])

# 搭建数据处理管线，对分类列进行转化
# OneHotEncoder参数解释：
# handle_unknown = 'ignore': 如果遇到未知类别，则忽略
# drop = 'first': 删除第一列，避免多重共线性
# sparse_output=False: 返回稀疏数组
cat_transformer = Pipeline(steps = [
    ("imputer", SimpleImputer(strategy = "most_frequent")),
    ("scaler", OneHotEncoder(handle_unknown = 'ignore', drop = 'first', sparse_output=False))
])

# 将两种类型的处理方式拼接起来，为后续处理数据集做准备
processor = ColumnTransformer(transformers = 
    [("num", num_transformer, numeric_features),
     ("cat", cat_transformer, category_features)
    ],
    remainder = "passthrough"
)

### 第四步：使用逻辑回归模型，测试不同的参数组合

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as stats

# 搭建建造逻辑回归模型的关系管线
clf_pipeline = Pipeline(steps = [
    ("processor", processor),
    ("classifier", LogisticRegression(solver="liblinear",random_state = 42))
]
)

# 预设逻辑回归模型的参数组合，使用RandomizedSearchCV来探索模型组合的可能性
param_dist = {
    'classifier__C': stats.loguniform(0.001, 100),
    'classifier__class_weight':[None, 'balanced']
}

# 参数说明：
# n_iter: 随机搜索迭代次数，即随机选择20组参数进行训练
# cv: 交叉验证折数，即5折交叉验证
# scoring: 评估指标，即使用roc_auc作为评估指标
# roc_auc:  ROC曲线下面积，即衡量模型分类性能的指标
# random_state: 随机种子，即保证随机搜索结果的可复现性
# n_jobs: 并行任务数，即使用所有可用的CPU核心
random_search = RandomizedSearchCV(
    clf_pipeline,
    param_distributions=param_dist,
    n_iter = 20,
    cv = 5, 
    scoring = "roc_auc",
    random_state = 42,
    n_jobs = -1
)

# 进行数据训练，找到最佳的参数组合
random_search.fit(X_train, y_train)
best_model = random_search.best_estimator_
print(best_model)
print(f"最佳参数组合是{random_search.best_params_}, 最佳的分是{random_search.best_score_}")

Pipeline(steps=[('processor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('imupter',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['SeniorCitizen', 'tenure',
                                                   'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                

### 步骤5：商业级评估 (ROC-AUC 与报告)

In [9]:
from sklearn.metrics import roc_auc_score, classification_report

print("测试集效果评估")
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:,1]

print("分类报告：\n", classification_report(y_test, y_pred))
print(f"AUC得分: {roc_auc_score(y_test, y_proba):.4f}")




测试集效果评估
分类报告：
               precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.65      0.55      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.79      0.80      0.80      1409

AUC得分: 0.8409


In [23]:
import pandas as pd

print("\n--> 步骤 7: 商业洞察 (导致流失的核心诱因)")
# 提取处理后的特征名
preprocessor_fitted = best_model.named_steps['processor']
cat_features = preprocessor_fitted.named_transformers_['cat'].named_steps['scaler'].get_feature_names_out(category_features)
# all_feature_names = preprocessor_fitted.get_feature_names_out()
all_feature_names_1 = numeric_features + list(cat_features)
# print(all_feature_names,"\n")
print(all_feature_names_1)

# 提取逻辑回归的权重
weights = best_model.named_steps['classifier'].coef_[0]

importance_df = pd.DataFrame({'Feature': all_feature_names_1, 'Weight': weights})
importance_df['Abs_Weight'] = importance_df['Weight'].abs()
importance_df = importance_df.sort_values(by='Abs_Weight', ascending=False)

print(importance_df.head(5)) # 打印 Top 5 核心因素


--> 步骤 7: 商业洞察 (导致流失的核心诱因)
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']
                           Feature    Weight  Abs_Weight
10     InternetService_Fiber optic  2.778445    2.778445
2                   MonthlyCharges -2.397415    2.397415
8   MultipleLines_No phone service -1.7902

### 模型序列化

In [24]:
import joblib
print("\n-->: 模型序列化")
joblib.dump(best_model, "best_model.pkl")
print("\n-->: 模型已保存为 best_model.pkl")



-->: 模型序列化

-->: 模型已保存为 best_model.pkl


### 编写 Streamlit 交互网页
详见app.py文件